# 🛡️ Netra-One Forensics: Google Colab Training & Evaluation Notebook
### **Forensic Image Verification & Digital Tamper Detection**
*Bharatiya Sakshya Adhiniyam (BSA), 2023 Section 63 Digital Evidence Standards*

This self-contained notebook performs end-to-end training and evaluation of the **Netra-One Forensic Classifier** (MobileNetV2 backbone) utilizing **Error Level Analysis (ELA)** and **High-Pass Noise Residual Maps** to detect image splicing, copy-move forgery, and synthetic manipulation in surveillance imagery.

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install dependencies in Google Colab
!pip install -q opencv-python pillow torch torchvision scipy matplotlib scikit-learn pandas tqdm

## 2. Imports & GPU Accelerator Check

In [ ]:
import os
import io
import sys
import json
import time
import random
import hashlib
import datetime
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageChops
import matplotlib.pyplot as plt
from scipy.fftpack import dct

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ Running on Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")

## 3. Dataset Acquisition / Benchmark Generation
Generates a balanced surveillance camera benchmark dataset (authentic vs tampered images) with realistic compression artifacts.

In [ ]:
def generate_surveillance_frame(width=640, height=480, scene_type="street"):
    img = np.zeros((height, width, 3), dtype=np.uint8)
    if scene_type == "street":
        for y in range(height):
            val = int(50 + 40 * (y / height) + random.randint(-5, 5))
            img[y, :] = (val, val, val)
        cv2.line(img, (width // 3, height), (width // 2 - 20, height // 2), (200, 200, 200), 4)
        cv2.line(img, (2 * width // 3, height), (width // 2 + 20, height // 2), (200, 200, 200), 4)
        cv2.rectangle(img, (0, 0), (width, height // 3), (120, 110, 100), -1)
    else:
        for y in range(0, height, 40):
            for x in range(0, width, 40):
                c = 180 + (x // 40 + y // 40) % 2 * 30 + random.randint(-4, 4)
                img[y:y+40, x:x+40] = (c, c - 10, c - 20)
    noise = np.random.normal(0, 8, (height, width, 3)).astype(np.float32)
    img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    cv2.putText(img, "NETRA-CAM-01 | 2026-09-29 18:22:04", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

def tamper_image(image_np, tamper_type="splice"):
    tampered = image_np.copy()
    h, w, _ = image_np.shape
    rw, rh = random.randint(int(w * 0.15), int(w * 0.35)), random.randint(int(h * 0.15), int(h * 0.35))
    rx, ry = random.randint(50, w - rw - 50), random.randint(50, h - rh - 50)
    
    if tamper_type == "splice":
        foreign = generate_surveillance_frame(w, h, scene_type="indoor")
        _, encoded = cv2.imencode('.jpg', foreign, [int(cv2.IMWRITE_JPEG_QUALITY), random.choice([55, 65, 75])])
        foreign_compressed = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
        tampered[ry:ry+rh, rx:rx+rw] = foreign_compressed[ry:ry+rh, rx:rx+rw]
    elif tamper_type == "copy_move":
        src_x = (rx + rw + 50) % (w - rw - 10)
        patch = cv2.GaussianBlur(tampered[ry:ry+rh, src_x:src_x+rw].copy(), (5, 5), 0)
        tampered[ry:ry+rh, rx:rx+rw] = patch
    elif tamper_type == "inpaint":
        tampered[ry:ry+rh, rx:rx+rw] = cv2.GaussianBlur(tampered[ry:ry+rh, rx:rx+rw], (21, 21), 0)
        
    _, final_encoded = cv2.imencode('.jpg', tampered, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    return cv2.imdecode(final_encoded, cv2.IMREAD_COLOR)

# Generate Dataset Directories
dataset_dir = "./data/surveillance_colab"
auth_dir = os.path.join(dataset_dir, "authentic")
tamp_dir = os.path.join(dataset_dir, "tampered")
os.makedirs(auth_dir, exist_ok=True)
os.makedirs(tamp_dir, exist_ok=True)

print(" Generating 100 Authentic and 100 Tampered Surveillance Benchmark Images...")
random.seed(42)
for i in range(100):
    frame = generate_surveillance_frame(640, 480, scene_type=random.choice(["street", "indoor"]))
    cv2.imwrite(os.path.join(auth_dir, f"auth_{i+1:03d}.jpg"), frame, [int(cv2.IMWRITE_JPEG_QUALITY), random.choice([88, 90, 92])])
    
    tframe = generate_surveillance_frame(640, 480, scene_type=random.choice(["street", "indoor"]))
    tamp_img = tamper_image(tframe, tamper_type=random.choice(["splice", "copy_move", "inpaint"]))
    cv2.imwrite(os.path.join(tamp_dir, f"tamp_{i+1:03d}.jpg"), tamp_img, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

print(" Benchmark Dataset Ready!")

## 4. Forensic Feature Extraction Modules
- **Error Level Analysis (ELA):** Re-saves image at JPEG Q=90 and measures compression delta.
- **Noise Residual Analysis:** High-pass spatial filtering for PRNU camera fingerprint inconsistencies.
- **Anomaly Scoring:** Robust Z-score computation across ELA block statistics.

In [ ]:
def extract_ela(img_rgb, quality=90, scale=15.0):
    pil_img = Image.fromarray(img_rgb).convert("RGB")
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    recompressed = Image.open(buf).convert("RGB")
    diff = ImageChops.difference(pil_img, recompressed)
    ela_diff_raw = np.array(diff, dtype=np.float32)
    ela_scaled = np.clip(ela_diff_raw * scale, 0, 255).astype(np.uint8)
    ela_gray = cv2.cvtColor(ela_scaled, cv2.COLOR_RGB2GRAY)
    ela_heatmap = cv2.applyColorMap(ela_gray, cv2.COLORMAP_JET)
    return ela_scaled, ela_heatmap, ela_diff_raw

def extract_noise_residual(img_rgb, ksize=5, block_size=32):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.medianBlur(gray, ksize)
    residual = cv2.absdiff(gray, blurred)
    h, w = gray.shape
    variance_map = np.zeros((h, w), dtype=np.float32)
    for y in range(0, h - block_size + 1, block_size):
        for x in range(0, w - block_size + 1, block_size):
            variance_map[y:y+block_size, x:x+block_size] = np.var(residual[y:y+block_size, x:x+block_size])
    max_v = variance_map.max()
    var_norm = variance_map / max_v if max_v > 0 else variance_map
    var_heatmap = cv2.applyColorMap((var_norm * 255).astype(np.uint8), cv2.COLORMAP_HOT)
    return residual, var_norm, var_heatmap

def compute_anomaly_score(ela_diff_raw, block_size=32):
    gray_diff = ela_diff_raw.mean(axis=2)
    h, w = gray_diff.shape
    means = []
    for y in range(0, h - block_size + 1, block_size):
        for x in range(0, w - block_size + 1, block_size):
            means.append(np.mean(gray_diff[y:y+block_size, x:x+block_size]))
    if not means: return 0.0
    means = np.array(means)
    med = np.median(means)
    mad = np.median(np.abs(means - med))
    score = (np.max(means) - med) / (1.4826 * mad) if mad > 1e-6 else np.max(means) - med
    return round(float(np.clip(score, 0.0, 99.99)), 2)

## 5. PyTorch Dataset & NetraForensicClassifier Architecture

In [ ]:
class ForensicDataset(Dataset):
    def __init__(self, samples_list, in_channels=3, quality=90):
        self.samples = samples_list
        self.in_channels = in_channels
        self.quality = quality
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_bgr = cv2.imread(path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ela_scaled, _, _ = extract_ela(img_rgb, quality=self.quality)
        tensor_input = self.transform(ela_scaled)
        return tensor_input, torch.tensor(label, dtype=torch.long)

class NetraForensicClassifier(nn.Module):
    def __init__(self, in_channels=3, dropout_rate=0.3):
        super(NetraForensicClassifier, self).__init__()
        base_model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.features = base_model.features
        self.pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(base_model.last_channel, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate / 2.0),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        f = self.features(x)
        p = self.pooling(f)
        return self.classifier(torch.flatten(p, 1))
    def predict_probability(self, x):
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)
            return torch.softmax(logits, dim=1)[:, 1].item()

## 6. Model Training & Validation Loop

In [ ]:
samples = []
for f in os.listdir(auth_dir): samples.append((os.path.join(auth_dir, f), 0))
for f in os.listdir(tamp_dir): samples.append((os.path.join(tamp_dir, f), 1))
random.shuffle(samples)

val_split = int(0.2 * len(samples))
train_samples, val_samples = samples[val_split:], samples[:val_split]

train_loader = DataLoader(ForensicDataset(train_samples), batch_size=16, shuffle=True)
val_loader = DataLoader(ForensicDataset(val_samples), batch_size=16, shuffle=False)

model = NetraForensicClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

epochs = 5
best_auc = 0.0
best_weights = "best_model.pth"

print(f" Starting Model Training for {epochs} Epochs...")
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    
    model.eval()
    y_true, y_probs, y_preds = [], [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(targets.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())
            y_preds.extend(preds.cpu().numpy())
            
    val_acc = accuracy_score(y_true, y_preds)
    val_auc = roc_auc_score(y_true, y_probs)
    print(f"Epoch {epoch:02d} | Train Loss: {running_loss/len(train_samples):.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
    if val_auc >= best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), best_weights)
        print(f" Saved checkpoint ({best_weights})")

## 7. Performance Evaluation & ROC Visualization

In [ ]:
model.load_state_dict(torch.load(best_weights))
model.eval()
y_true, y_probs, y_preds = [], [], []
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)
        y_true.extend(targets.numpy())
        y_probs.extend(probs.cpu().numpy())
        y_preds.extend(preds.cpu().numpy())

acc = accuracy_score(y_true, y_preds)
prec = precision_score(y_true, y_preds)
rec = recall_score(y_true, y_preds)
f1 = f1_score(y_true, y_preds)
auc = roc_auc_score(y_true, y_probs)

print("\n" + "="*40)
print(f" Accuracy  : {acc:.4f}")
print(f" Precision : {prec:.4f}")
print(f" Recall    : {rec:.4f}")
print(f" F1 Score  : {f1:.4f}")
print(f" ROC-AUC   : {auc:.4f}")
print("="*40)

fpr, tpr, _ = roc_curve(y_true, y_probs)
plt.figure(figsize=(6, 5), dpi=120)
plt.plot(fpr, tpr, color='blue', lw=2, label=f'MobileNetV2 (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('Netra-One ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Forensic Audit Demo & Evidence JSON Generation

In [ ]:
test_image_path = os.path.join(tamp_dir, os.listdir(tamp_dir)[0])
sha256_checksum = hashlib.sha256(open(test_image_path, 'rb').read()).hexdigest()

img_bgr = cv2.imread(test_image_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
ela_scaled, ela_heatmap, ela_diff_raw = extract_ela(img_rgb)
_, _, noise_heatmap = extract_noise_residual(img_rgb)
anomaly_score = compute_anomaly_score(ela_diff_raw)

tf = transforms.Compose([transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
inp_t = tf(ela_scaled).unsqueeze(0).to(device)
prob = model.predict_probability(inp_t)
verdict = "FLAGGED_TAMPERED" if prob >= 0.50 else "AUTHENTIC"

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
axes[0].imshow(img_rgb); axes[0].set_title("Original Surveillance Image"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(ela_heatmap, cv2.COLOR_BGR2RGB)); axes[1].set_title("ELA Difference Heatmap"); axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(noise_heatmap, cv2.COLOR_BGR2RGB)); axes[2].set_title("Noise Residual Map"); axes[2].axis("off")
plt.suptitle(f"Verdict: {verdict} | P(Tamper) = {prob:.2f} | Anomaly Score = {anomaly_score}", color="red" if verdict=="FLAGGED_TAMPERED" else "green", fontsize=14, fontweight="bold")
plt.show()

report = {
    "file_name": os.path.basename(test_image_path),
    "sha256": sha256_checksum,
    "tamper_verdict": verdict,
    "manipulation_probability": round(float(prob), 4),
    "compression_artifact_anomaly_score": anomaly_score,
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "legal_standard": "Bharatiya Sakshya Adhiniyam 2023 Section 63 Compliant"
}
print(json.dumps(report, indent=2))

## 9. Download Trained Model Weights

In [ ]:
try:
    from google.colab import files
    files.download('best_model.pth')
except Exception:
    print(" Trained weights saved as 'best_model.pth' in working directory.")